<a href="https://colab.research.google.com/github/mdzubayereashraf/Ostad-AI-Engineering-Course-Assignments/blob/main/Assignmen_No_19(TeleSupport_AI_Zubayer).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **TeleSupport AI – LLM Structured Extraction & Tool-Calling Assistant**

## **Introduction**

Large Language Models (LLMs) have transformed the field of Artificial Intelligence by enabling computers to understand and generate human language. However, modern AI applications require more than just generating text. They also need to extract structured information and interact with external tools to perform useful tasks.

In this assignment, I will develop a prototype called TeleSupport AI, designed to assist customers of a telecommunications company.

The project consists of two main components:

**1. Structured Customer Data Extraction:** This component uses Pydantic and an LLM to extract customer information from unstructured text messages and convert it into validated JSON data.

**2. Tool-Calling AI Assistant:** This component uses LangChain to create an interactive AI assistant capable of dynamically calling Python functions to retrieve account balances, find internet packages, create support tickets, and check network status.

Additionally, I will compare cloud-based and locally hosted LLMs using Ollama to understand their differences in performance, processing latency, and extraction accuracy.

The overall goal is to gain practical experience with structured outputs, LLM integration, function calling, and AI agent development.

### **Verify the API Key**

In [1]:
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

if GEMINI_API_KEY:
    print("Gemini API key loaded successfully!")
else:
    print("API key not found!")

Gemini API key loaded successfully!


### **Install Required Libraries**

In [2]:
!pip install -q openai pydantic

### **Import Libraries**

In [3]:
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Literal
from google.colab import userdata
import json
import time

### **Connect Gemini API**

In [9]:
client = OpenAI(
    api_key=GEMINI_API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    max_retries=5,
    timeout=60.0
)

MODEL_NAME = "gemini-3.8-flash"

In [11]:
try:
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "user",
                "content": "Say hello in one sentence."
            }
        ]
    )

    print("Gemini connected successfully!")
    print(response.choices[0].message.content)

except Exception as e:
    print("Connection failed:", e)

Connection failed: Error code: 503 - [{'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}]


### **Pydantic Schema**

In [12]:
from pydantic import BaseModel, Field
from typing import Literal
import json


class CustomerTicket(BaseModel):

    name: str = Field(
        ...,
        description="The name of the customer. Use 'Unknown' if not explicitly stated."
    )

    phone: str = Field(
        ...,
        description="The customer's phone number. Use 'Unknown' if not provided."
    )

    location: str = Field(
        ...,
        description="The physical location or area mentioned by the customer. Use 'Unknown' if not provided."
    )

    category: Literal[
        "BALANCE",
        "INTERNET",
        "RECHARGE",
        "SIM",
        "COMPLAINT",
        "OTHER"
    ]

    sentiment: Literal[
        "Positive",
        "Negative",
        "Neutral"
    ]

    urgency: int = Field(
        ...,
        ge=1,
        le=5,
        description="Urgency scale from 1 (lowest) to 5 (highest/critical)."
    )

    summary: str = Field(
        ...,
        description="A concise one-sentence summary of the user message."
    )

### **Generate JSON Schema**

In [13]:
print(json.dumps(
    CustomerTicket.model_json_schema(),
    indent=4
))

{
    "properties": {
        "name": {
            "description": "The name of the customer. Use 'Unknown' if not explicitly stated.",
            "title": "Name",
            "type": "string"
        },
        "phone": {
            "description": "The customer's phone number. Use 'Unknown' if not provided.",
            "title": "Phone",
            "type": "string"
        },
        "location": {
            "description": "The physical location or area mentioned by the customer. Use 'Unknown' if not provided.",
            "title": "Location",
            "type": "string"
        },
        "category": {
            "enum": [
                "BALANCE",
                "INTERNET",
                "RECHARGE",
                "SIM",
                "COMPLAINT",
                "OTHER"
            ],
            "title": "Category",
            "type": "string"
        },
        "sentiment": {
            "enum": [
                "Positive",
                "Negative",
           

### **Test Pydantic validation**

In [14]:
ticket = CustomerTicket(
    name="Sayem",
    phone="01712345678",
    location="Uttara",
    category="COMPLAINT",
    sentiment="Negative",
    urgency=4,
    summary="Customer reports a network outage in Uttara."
)

print(ticket.model_dump_json(indent=4))

{
    "name": "Sayem",
    "phone": "01712345678",
    "location": "Uttara",
    "category": "COMPLAINT",
    "sentiment": "Negative",
    "urgency": 4,
    "summary": "Customer reports a network outage in Uttara."
}
